# Scheme 3 - Lax-Wendroff/Richtmyer-style flux for advection + diffusion
## Load Configuration and Dependencies
Import core libraries, define all dataclasses, helpers, kernels, and solvers inline.
This notebook is self-contained and runs all experiments end-to-end. Execute each section in order;
the final section iterates over the full experiment catalogue, writes summary CSVs, and saves plots to the notebook folder.

## Scheme 3 Equations and Discretization

### Governing PDEs
- Linear advection-diffusion: $u_t + c\,u_x = \nu\,u_{xx}$
- Viscous Burgers: $u_t + \left(\tfrac{u^2}{2}\right)_x = \nu\,u_{xx}$

### Spatial Discretization and Time Update (Lax–Wendroff)
Let $x_i = i\,\Delta x$, $u_i^n \approx u(x_i, t^n)$.

#### Linear Advection-Diffusion (Lax–Wendroff flux)
- Numerical flux at $i+1/2$:
  $f_{i+1/2} = \tfrac{c}{2}(u_i + u_{i+1}) - \tfrac{c^2\,\Delta t}{2\,\Delta x}(u_{i+1} - u_i)$
- Diffusion second derivative:
  $u_{xx} \approx \dfrac{u_{i+1}^n - 2u_i^n + u_{i-1}^n}{\Delta x^2}$
- Update:
  $u_i^{n+1} = u_i^n - \Delta t\,\dfrac{f_{i+1/2} - f_{i-1/2}}{\Delta x} + \nu\,\Delta t\,u_{xx}$

#### Viscous Burgers (Lax–Wendroff / Richtmyer two-step flux)
- Physical flux: $f(u) = \tfrac{1}{2}u^2$
- Predictor (half-step state at $i+1/2$):
  $u_{i+1/2}^{n+1/2} = \tfrac{1}{2}(u_i + u_{i+1}) - \dfrac{\Delta t}{2\,\Delta x}\bigl(f(u_{i+1}) - f(u_i)\bigr)$
- Corrector flux:
  $f_{i+1/2} = f\!\left(u_{i+1/2}^{n+1/2}\right) = \tfrac{1}{2}\left(u_{i+1/2}^{n+1/2}\right)^2$
- Diffusion second derivative as above
- Update:
  $u_i^{n+1} = u_i^n - \Delta t\,\dfrac{f_{i+1/2} - f_{i-1/2}}{\Delta x} + \nu\,\Delta t\,u_{xx}$

### CFL Time Step (Combined Advection + Diffusion)
- Linear: $\Delta t = \dfrac{\mathrm{CFL}}{|c|/\Delta x + 2\nu/\Delta x^2}$
- Burgers: $\Delta t = \dfrac{\mathrm{CFL}}{|u|_{\max}/\Delta x + 2\nu/\Delta x^2}$


## 1. Load Dependencies and Paths
Import all required libraries and configure the output directory and plot defaults.
Prints library versions so the environment is fully reproducible.


In [ ]:
import math
import time
import traceback
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


OUTDIR = Path.cwd()
OUTDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.grid"] = True

DTYPE_MAP = {
    "float64": torch.float64,
    "float32": torch.float32,
    "float16": torch.float16,
}


## 2. Define ExperimentConfig and RunResult
Define `ExperimentConfig` (experiment parameters) and `RunResult` (per-run diagnostics).
Also defines the `DTYPE_MAP` mapping precision name strings to `torch.dtype` values.


In [ ]:
@dataclass
class ExperimentConfig:
    name: str
    description: str

    nx: int = 1024

    tfinal_linear: float = 0.20
    c_linear: float = 1.0
    nu_linear: float = 1.0e-3
    linear_mode: int = 1
    linear_amplitude: float = 1.0

    tfinal_burgers: float = 0.12
    re_values: List[float] = field(default_factory=lambda: [10.0, 100.0, 1000.0])

    cfl_total: float = 0.80

    prefer_cuda: bool = False

    run_linear: bool = True
    run_burgers: bool = True

    run_stability_sweep: bool = True
    stability_re_for_burgers: float = 1000.0
    stability_cfl_candidates: List[float] = field(
        default_factory=lambda: [
            0.10, 0.20, 0.30, 0.40, 0.50,
            0.60, 0.70, 0.80, 0.90, 1.00,
            1.10, 1.20, 1.30, 1.40,
        ]
    )
    stability_amplitude_limit: float = 10.0

    audit_dtypes: bool = True

    dense_reference_nx: int = 8192


@dataclass
class RunResult:
    equation: str
    precision: str
    x: np.ndarray
    u0: np.ndarray
    u_final: np.ndarray
    u_ref: np.ndarray
    dx: float
    tfinal: float
    dt_last: float
    dt_min: float
    dt_max: float
    nsteps: int
    runtime_sec: float
    l2_error: float
    linf_error: float
    amplitude_error: Optional[float]
    phase_error: Optional[float]
    conservation_error_sum: float
    mass_initial: float
    mass_final: float
    mass_drift: float
    tv_initial: float
    tv_final: float
    stable: bool
    notes: str
    state_bytes: int
    est_flops_total: float
    est_bytes_total: float
    re_value: Optional[float] = None
    reference_type: str = ""


## 3. Device, Tensor, and Diagnostics Helpers
Utility functions for device selection, tensor construction at a given dtype, numpy conversion,
dtype assertion checks, mass/TV/error diagnostics, and estimated FLOP/byte counts.


In [ ]:
def pick_device(cfg: ExperimentConfig) -> torch.device:
    if cfg.prefer_cuda and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def tconst(x, dtype: torch.dtype, device: torch.device) -> torch.Tensor:
    return torch.tensor(x, dtype=dtype, device=device)


def to_numpy(u: torch.Tensor) -> np.ndarray:
    return u.detach().to(torch.float64).cpu().numpy()


def assert_dtype(u: torch.Tensor, dtype: torch.dtype, name: str) -> None:
    if u.dtype != dtype:
        raise TypeError(f"{name} has dtype {u.dtype}, expected {dtype}")


def compute_mass(u: torch.Tensor) -> float:
    return float(u.detach().to(torch.float64).mean().cpu())


def compute_sum(u: torch.Tensor) -> float:
    return float(u.detach().to(torch.float64).sum().cpu())


def total_variation(u: torch.Tensor) -> float:
    u64 = u.detach().to(torch.float64)
    return float(torch.sum(torch.abs(torch.roll(u64, shifts=-1, dims=0) - u64)).cpu())


def l2_error_np(u: np.ndarray, u_ref: np.ndarray) -> float:
    diff = u.astype(np.float64) - u_ref.astype(np.float64)
    return float(np.sqrt(np.mean(diff * diff)))


def linf_error_np(u: np.ndarray, u_ref: np.ndarray) -> float:
    diff = np.abs(u.astype(np.float64) - u_ref.astype(np.float64))
    return float(np.max(diff))


def amplitude_of_signal(u: np.ndarray) -> float:
    u64 = u.astype(np.float64)
    return 0.5 * (float(np.max(u64)) - float(np.min(u64)))


def amplitude_error_np(u: np.ndarray, u_ref: np.ndarray) -> float:
    return abs(amplitude_of_signal(u) - amplitude_of_signal(u_ref))


def phase_error_np(x: np.ndarray, u: np.ndarray, u_ref: np.ndarray) -> float:
    a = u.astype(np.float64) - np.mean(u.astype(np.float64))
    b = u_ref.astype(np.float64) - np.mean(u_ref.astype(np.float64))
    corr = np.fft.ifft(np.fft.fft(a) * np.conj(np.fft.fft(b)))
    shift = int(np.argmax(np.abs(corr)))
    n = len(x)
    if shift > n // 2:
        shift -= n
    dx = float(x[1] - x[0])
    return abs(shift) * dx


def estimate_costs(equation: str, nx: int, nsteps: int, itemsize: int) -> Tuple[float, float]:
    if equation == "linear_advection_diffusion":
        flops_per_cell = 24.0
        arrays_touched_per_step = 9.0
    else:
        flops_per_cell = 48.0
        arrays_touched_per_step = 15.0

    est_flops_total = flops_per_cell * nx * nsteps
    est_bytes_total = arrays_touched_per_step * nx * itemsize * nsteps
    return est_flops_total, est_bytes_total


## 4. Initial Conditions and Reference Builders
Defines sinusoidal initial conditions for both equations, the exact linear solution,
and the Cole–Hopf spectral reference for viscous Burgers.


In [ ]:
def initial_linear_condition_torch(
    x: torch.Tensor,
    mode: int,
    amp: float,
    dtype: torch.dtype,
    device: torch.device,
) -> torch.Tensor:
    two_pi = tconst(2.0 * math.pi, dtype, device)
    return tconst(amp, dtype, device) * torch.sin(two_pi * tconst(float(mode), dtype, device) * x)


def initial_burgers_condition_torch(
    x: torch.Tensor,
    dtype: torch.dtype,
    device: torch.device,
) -> torch.Tensor:
    two_pi = tconst(2.0 * math.pi, dtype, device)
    return torch.sin(two_pi * x)


def exact_linear_solution(
    x: np.ndarray,
    t: float,
    c: float,
    nu: float,
    mode: int = 1,
    amp: float = 1.0,
) -> np.ndarray:
    k = 2.0 * np.pi * mode
    return amp * np.exp(-nu * (k ** 2) * t) * np.sin(k * (x - c * t))


def burgers_reference_cole_hopf(
    x_target: np.ndarray,
    t: float,
    nu: float,
    dense_nx: int = 8192,
) -> np.ndarray:
    x_dense = np.linspace(0.0, 1.0, dense_nx, endpoint=False)

    exponent = (np.cos(2.0 * np.pi * x_dense) - 1.0) / (4.0 * np.pi * nu)
    phi0 = np.exp(exponent)

    k = 2.0 * np.pi * np.fft.fftfreq(dense_nx, d=1.0 / dense_nx)
    phi_hat = np.fft.fft(phi0)
    decay = np.exp(-nu * (k ** 2) * t)

    phi_t = np.fft.ifft(phi_hat * decay).real
    dphi_t = np.fft.ifft((1j * k) * phi_hat * decay).real

    scale = max(float(np.max(np.abs(phi_t))), 1.0)
    eps = max(np.finfo(np.float64).eps * scale, 1.0e-300)
    phi_safe = np.where(np.abs(phi_t) < eps, np.where(phi_t >= 0.0, eps, -eps), phi_t)

    u_dense = -2.0 * nu * dphi_t / phi_safe
    u_dense = np.nan_to_num(u_dense, nan=0.0, posinf=0.0, neginf=0.0)

    x_aug = np.concatenate([x_dense, [1.0]])
    u_aug = np.concatenate([u_dense, [u_dense[0]]])

    return np.interp(x_target, x_aug, u_aug)


## 5. Time-Step Rules and Scheme 3 Kernels
CFL-based time-step formulae and the Lax–Wendroff update kernels for both equations.
Linear uses the standard Lax–Wendroff flux; Burgers uses the Richtmyer two-step form.


In [ ]:
def dt_linear_combined(c: float, nu: float, dx: float, cfl_total: float = 0.80) -> float:
    denom = abs(c) / dx + 2.0 * nu / (dx * dx)
    return float(cfl_total / denom)


def dt_burgers_combined(umax: float, nu: float, dx: float, cfl_total: float = 0.80) -> float:
    umax = max(float(umax), 1.0e-14)
    denom = umax / dx + 2.0 * nu / (dx * dx)
    return float(cfl_total / denom)


def linear_lw_flux(
    u: torch.Tensor,
    c: float,
    dt: float,
    dx: float,
    dtype: torch.dtype,
    device: torch.device,
    audit_dtypes: bool,
) -> torch.Tensor:
    half_t = tconst(0.5, dtype, device)
    c_t = tconst(c, dtype, device)
    dt_t = tconst(dt, dtype, device)
    dx_t = tconst(dx, dtype, device)

    u_right = torch.roll(u, shifts=-1, dims=0)
    flux = half_t * c_t * (u + u_right) - half_t * c_t * c_t * (dt_t / dx_t) * (u_right - u)
    flux = flux.to(dtype)

    if audit_dtypes:
        assert_dtype(flux, dtype, "linear_lw_flux")

    return flux


def linear_update(
    u: torch.Tensor,
    c: float,
    nu: float,
    dt: float,
    dx: float,
    dtype: torch.dtype,
    device: torch.device,
    audit_dtypes: bool,
) -> torch.Tensor:
    dt_t = tconst(dt, dtype, device)
    dx_t = tconst(dx, dtype, device)
    nu_t = tconst(nu, dtype, device)
    two_t = tconst(2.0, dtype, device)

    flux_iphalf = linear_lw_flux(u, c, dt, dx, dtype, device, audit_dtypes)
    flux_imhalf = torch.roll(flux_iphalf, shifts=1, dims=0)

    dflux_dx = (flux_iphalf - flux_imhalf) / dx_t
    d2u_dx2 = (torch.roll(u, shifts=-1, dims=0) - two_t * u + torch.roll(u, shifts=1, dims=0)) / (dx_t * dx_t)

    if audit_dtypes:
        assert_dtype(dflux_dx, dtype, "dflux_dx_linear")
        assert_dtype(d2u_dx2, dtype, "d2u_dx2_linear")

    u_new = u - dt_t * dflux_dx + nu_t * dt_t * d2u_dx2
    u_new = u_new.to(dtype)

    if audit_dtypes:
        assert_dtype(u_new, dtype, "u_new_linear")

    return u_new


def physical_flux_burgers(
    u: torch.Tensor,
    dtype: torch.dtype,
    device: torch.device,
) -> torch.Tensor:
    half_t = tconst(0.5, dtype, device)
    return (half_t * u * u).to(dtype)


def lax_wendroff_flux_burgers(
    u_left: torch.Tensor,
    u_right: torch.Tensor,
    dt: float,
    dx: float,
    dtype: torch.dtype,
    device: torch.device,
    audit_dtypes: bool,
) -> torch.Tensor:
    half_t = tconst(0.5, dtype, device)
    dt_t = tconst(dt, dtype, device)
    dx_t = tconst(dx, dtype, device)

    f_left = physical_flux_burgers(u_left, dtype, device)
    f_right = physical_flux_burgers(u_right, dtype, device)
    u_half = half_t * (u_left + u_right) - half_t * (dt_t / dx_t) * (f_right - f_left)
    u_half = u_half.to(dtype)
    flux = physical_flux_burgers(u_half, dtype, device)

    if audit_dtypes:
        assert_dtype(flux, dtype, "lax_wendroff_flux_burgers")

    return flux


def burgers_update(
    u: torch.Tensor,
    nu: float,
    dt: float,
    dx: float,
    dtype: torch.dtype,
    device: torch.device,
    audit_dtypes: bool,
) -> torch.Tensor:
    dt_t = tconst(dt, dtype, device)
    dx_t = tconst(dx, dtype, device)
    nu_t = tconst(nu, dtype, device)
    two_t = tconst(2.0, dtype, device)

    u_left = u
    u_right = torch.roll(u, shifts=-1, dims=0)
    flux_iphalf = lax_wendroff_flux_burgers(u_left, u_right, dt, dx, dtype, device, audit_dtypes)
    flux_imhalf = torch.roll(flux_iphalf, shifts=1, dims=0)

    dflux_dx = (flux_iphalf - flux_imhalf) / dx_t
    d2u_dx2 = (torch.roll(u, shifts=-1, dims=0) - two_t * u + torch.roll(u, shifts=1, dims=0)) / (dx_t * dx_t)

    if audit_dtypes:
        assert_dtype(dflux_dx, dtype, "dflux_dx_burgers")
        assert_dtype(d2u_dx2, dtype, "d2u_dx2_burgers")

    u_new = u - dt_t * dflux_dx + nu_t * dt_t * d2u_dx2
    u_new = u_new.to(dtype)

    if audit_dtypes:
        assert_dtype(u_new, dtype, "u_new_burgers")

    return u_new


## 6. Baseline Generation for Linear and Burgers
Generate the exact linear solution and Burgers Cole–Hopf references for each experiment
and save them to disk as CSVs for reuse across solver runs.


In [ ]:
def build_baselines(cfg: ExperimentConfig, base_dir: Path):
    base_dir.mkdir(parents=True, exist_ok=True)

    x_np = np.linspace(0.0, 1.0, cfg.nx, endpoint=False)

    linear_exact = exact_linear_solution(
        x_np,
        cfg.tfinal_linear,
        cfg.c_linear,
        cfg.nu_linear,
        cfg.linear_mode,
        cfg.linear_amplitude,
    )
    pd.DataFrame({"x": x_np, "u_exact": linear_exact}).to_csv(
        base_dir / "linear_exact_baseline.csv", index=False
    )

    burgers_refs = {}
    if cfg.run_burgers:
        for re_value in cfg.re_values:
            nu = 1.0 / re_value
            u_ref = burgers_reference_cole_hopf(
                x_np,
                cfg.tfinal_burgers,
                nu,
                dense_nx=max(cfg.dense_reference_nx, cfg.nx * 8),
            )
            burgers_refs[re_value] = u_ref
            pd.DataFrame({"x": x_np, "u_reference": u_ref}).to_csv(
                base_dir / f"burgers_reference_Re_{int(re_value)}.csv", index=False
            )

    return x_np, linear_exact, burgers_refs


## 7. Linear Advection-Diffusion Runner
Run the linear solver in FP64/FP32/FP16 for a given experiment, collect L2/Linf/amplitude/phase
errors, conservation, TV, runtime, and memory metrics, and return a `RunResult`.


In [ ]:
def run_linear_advection_diffusion(
    precision_name: str,
    cfg: ExperimentConfig,
    device: torch.device,
    u_ref_linear: np.ndarray,
) -> RunResult:
    dtype = DTYPE_MAP[precision_name]

    x_np = np.linspace(0.0, 1.0, cfg.nx, endpoint=False)
    dx = float(x_np[1] - x_np[0])

    x_t = torch.tensor(x_np, dtype=dtype, device=device)
    u0_t = initial_linear_condition_torch(
        x_t, cfg.linear_mode, cfg.linear_amplitude, dtype, device
    )
    u_t = u0_t.clone()

    if cfg.audit_dtypes:
        assert_dtype(u_t, dtype, "u_linear_initial")

    mass_initial = compute_mass(u_t)
    sum_initial = compute_sum(u_t)
    tv_initial = total_variation(u_t)

    dt = dt_linear_combined(cfg.c_linear, cfg.nu_linear, dx, cfg.cfl_total)
    nsteps = int(np.ceil(cfg.tfinal_linear / dt))
    dt = cfg.tfinal_linear / nsteps

    stable, notes = True, "OK"

    start = time.perf_counter()

    for _ in range(nsteps):
        u_t = linear_update(
            u_t,
            cfg.c_linear,
            cfg.nu_linear,
            dt,
            dx,
            dtype,
            device,
            cfg.audit_dtypes,
        )
        if not bool(torch.isfinite(u_t).all().item()):
            stable, notes = False, "Non-finite values detected"
            break

    if device.type == "cuda":
        torch.cuda.synchronize(device)

    runtime_sec = time.perf_counter() - start

    u_final_np = to_numpy(u_t)
    u0_np = to_numpy(u0_t)

    mass_final = compute_mass(u_t)
    sum_final = compute_sum(u_t)
    tv_final = total_variation(u_t)

    state_bytes = int(u_t.numel() * u_t.element_size())
    est_flops_total, est_bytes_total = estimate_costs(
        "linear_advection_diffusion", cfg.nx, nsteps, u_t.element_size()
    )

    return RunResult(
        equation="linear_advection_diffusion",
        precision=precision_name,
        x=x_np,
        u0=u0_np,
        u_final=u_final_np,
        u_ref=u_ref_linear.astype(np.float64),
        dx=dx,
        tfinal=cfg.tfinal_linear,
        dt_last=dt,
        dt_min=dt,
        dt_max=dt,
        nsteps=nsteps,
        runtime_sec=runtime_sec,
        l2_error=l2_error_np(u_final_np, u_ref_linear),
        linf_error=linf_error_np(u_final_np, u_ref_linear),
        amplitude_error=amplitude_error_np(u_final_np, u_ref_linear),
        phase_error=phase_error_np(x_np, u_final_np, u_ref_linear),
        conservation_error_sum=abs(sum_final - sum_initial),
        mass_initial=mass_initial,
        mass_final=mass_final,
        mass_drift=abs(mass_final - mass_initial),
        tv_initial=tv_initial,
        tv_final=tv_final,
        stable=stable,
        notes=notes,
        state_bytes=state_bytes,
        est_flops_total=est_flops_total,
        est_bytes_total=est_bytes_total,
        re_value=None,
        reference_type="exact_solution",
    )


## 8. Burgers Runner with FP64 Schedule
Build a stable FP64 pilot time-step schedule for each Reynolds number, then replay it at
FP64/FP32/FP16 to isolate precision effects from time-step differences.


In [ ]:
def build_burgers_dt_schedule_fp64(
    re_value: float,
    cfg: ExperimentConfig,
    device: torch.device,
):
    dtype = torch.float64
    nu = 1.0 / re_value

    x_np = np.linspace(0.0, 1.0, cfg.nx, endpoint=False)
    dx = float(x_np[1] - x_np[0])

    x_t = torch.tensor(x_np, dtype=dtype, device=device)
    u_t = initial_burgers_condition_torch(x_t, dtype, device)

    t = 0.0
    dt_schedule = []
    stable = True

    while t < cfg.tfinal_burgers - 1.0e-15:
        umax = float(torch.max(torch.abs(u_t)).detach().cpu())
        dt = dt_burgers_combined(umax, nu, dx, cfg.cfl_total)

        if t + dt > cfg.tfinal_burgers:
            dt = cfg.tfinal_burgers - t

        u_t = burgers_update(u_t, nu, dt, dx, dtype, device, False)

        if not bool(torch.isfinite(u_t).all().item()):
            stable = False
            break

        dt_schedule.append(float(dt))
        t += dt

    return dt_schedule, stable


def run_burgers_with_schedule(
    precision_name: str,
    re_value: float,
    dt_schedule: list[float],
    cfg: ExperimentConfig,
    device: torch.device,
    u_ref_burgers: np.ndarray,
) -> RunResult:
    dtype = DTYPE_MAP[precision_name]
    nu = 1.0 / re_value

    x_np = np.linspace(0.0, 1.0, cfg.nx, endpoint=False)
    dx = float(x_np[1] - x_np[0])

    x_t = torch.tensor(x_np, dtype=dtype, device=device)
    u0_t = initial_burgers_condition_torch(x_t, dtype, device)
    u_t = u0_t.clone()

    if cfg.audit_dtypes:
        assert_dtype(u_t, dtype, "u_burgers_initial")

    mass_initial = compute_mass(u_t)
    sum_initial = compute_sum(u_t)
    tv_initial = total_variation(u_t)

    stable, notes = True, "OK"

    start = time.perf_counter()
    dt_used = []

    for dt in dt_schedule:
        u_t = burgers_update(u_t, nu, dt, dx, dtype, device, cfg.audit_dtypes)
        dt_used.append(dt)

        if not bool(torch.isfinite(u_t).all().item()):
            stable, notes = False, "Non-finite values detected"
            break

    if device.type == "cuda":
        torch.cuda.synchronize(device)

    runtime_sec = time.perf_counter() - start

    u_final_np = to_numpy(u_t)
    u0_np = to_numpy(u0_t)

    mass_final = compute_mass(u_t)
    sum_final = compute_sum(u_t)
    tv_final = total_variation(u_t)

    state_bytes = int(u_t.numel() * u_t.element_size())
    est_flops_total, est_bytes_total = estimate_costs(
        "viscous_burgers", cfg.nx, len(dt_used), u_t.element_size()
    )

    return RunResult(
        equation="viscous_burgers",
        precision=precision_name,
        x=x_np,
        u0=u0_np,
        u_final=u_final_np,
        u_ref=u_ref_burgers.astype(np.float64),
        dx=dx,
        tfinal=cfg.tfinal_burgers,
        dt_last=dt_used[-1] if dt_used else 0.0,
        dt_min=min(dt_used) if dt_used else 0.0,
        dt_max=max(dt_used) if dt_used else 0.0,
        nsteps=len(dt_used),
        runtime_sec=runtime_sec,
        l2_error=l2_error_np(u_final_np, u_ref_burgers),
        linf_error=linf_error_np(u_final_np, u_ref_burgers),
        amplitude_error=None,
        phase_error=None,
        conservation_error_sum=abs(sum_final - sum_initial),
        mass_initial=mass_initial,
        mass_final=mass_final,
        mass_drift=abs(mass_final - mass_initial),
        tv_initial=tv_initial,
        tv_final=tv_final,
        stable=stable,
        notes=notes,
        state_bytes=state_bytes,
        est_flops_total=est_flops_total,
        est_bytes_total=est_bytes_total,
        re_value=re_value,
        reference_type="cole_hopf_reference",
    )


## 9. Stability Sweep Utilities
Estimate the largest stable CFL for linear and Burgers (when enabled) by scanning candidate
CFL values and checking for non-finite or blown-up solutions.


In [ ]:
def estimate_largest_stable_cfl_linear(
    precision_name: str,
    cfg: ExperimentConfig,
    device: torch.device,
    u_ref_linear: np.ndarray,
) -> float:
    best = 0.0
    for cfl in cfg.stability_cfl_candidates:
        cfg_trial = ExperimentConfig(**vars(cfg))
        cfg_trial.cfl_total = cfl
        res = run_linear_advection_diffusion(precision_name, cfg_trial, device, u_ref_linear)
        max_abs = float(np.max(np.abs(res.u_final)))
        if res.stable and np.isfinite(max_abs) and max_abs <= cfg.stability_amplitude_limit:
            best = cfl
        else:
            break
    return best


def estimate_largest_stable_cfl_burgers(
    precision_name: str,
    cfg: ExperimentConfig,
    device: torch.device,
    u_ref_burgers: np.ndarray,
) -> float:
    best = 0.0
    re_value = cfg.stability_re_for_burgers

    for cfl in cfg.stability_cfl_candidates:
        cfg_trial = ExperimentConfig(**vars(cfg))
        cfg_trial.cfl_total = cfl

        dt_schedule, pilot_stable = build_burgers_dt_schedule_fp64(
            re_value, cfg_trial, device
        )
        if not pilot_stable:
            break

        res = run_burgers_with_schedule(
            precision_name, re_value, dt_schedule, cfg_trial, device, u_ref_burgers
        )
        max_abs = float(np.max(np.abs(res.u_final)))
        if res.stable and np.isfinite(max_abs) and max_abs <= cfg.stability_amplitude_limit:
            best = cfl
        else:
            break

    return best


def save_stability_table(
    cfg: ExperimentConfig,
    device: torch.device,
    exp_dir: Path,
    u_ref_linear: np.ndarray,
    u_ref_burgers: Optional[np.ndarray],
) -> pd.DataFrame:
    rows = []

    if cfg.run_linear:
        for precision in ["float64", "float32", "float16"]:
            rows.append({
                "precision": precision,
                "equation": "linear_advection_diffusion",
                "largest_stable_cfl": estimate_largest_stable_cfl_linear(
                    precision, cfg, device, u_ref_linear
                ),
            })

    if cfg.run_burgers and u_ref_burgers is not None:
        for precision in ["float64", "float32", "float16"]:
            rows.append({
                "precision": precision,
                "equation": f"viscous_burgers_Re_{int(cfg.stability_re_for_burgers)}",
                "largest_stable_cfl": estimate_largest_stable_cfl_burgers(
                    precision, cfg, device, u_ref_burgers
                ),
            })

    df = pd.DataFrame(rows)
    df.to_csv(exp_dir / "stability_summary.csv", index=False)
    return df


## 10. Result Tables and CSV Writers
Build summary DataFrames for linear and Burgers results covering accuracy, conservation,
total variation, runtime, memory, and estimated computational cost. Saves per-experiment CSVs.


In [ ]:
def linear_summary_table(linear_results: Dict[str, RunResult]) -> pd.DataFrame:
    rows = []
    for precision in ["float64", "float32", "float16"]:
        r = linear_results[precision]
        rows.append({
            "equation": r.equation,
            "precision": r.precision,
            "reference_type": r.reference_type,
            "dt_last": r.dt_last,
            "nsteps": r.nsteps,
            "L2_error": r.l2_error,
            "Linf_error": r.linf_error,
            "amplitude_error": r.amplitude_error,
            "phase_error": r.phase_error,
            "conservation_error_sum": r.conservation_error_sum,
            "mass_drift": r.mass_drift,
            "TV_initial": r.tv_initial,
            "TV_final": r.tv_final,
            "runtime_sec": r.runtime_sec,
            "state_bytes": r.state_bytes,
            "memory_MB": r.state_bytes / (1024.0 ** 2),
            "est_flops_total": r.est_flops_total,
            "est_bytes_total": r.est_bytes_total,
            "stable": r.stable,
            "notes": r.notes,
        })
    return pd.DataFrame(rows)


def burgers_summary_table(
    burgers_results: Dict[float, Dict[str, RunResult]]
) -> pd.DataFrame:
    rows = []
    for re_value, runs in burgers_results.items():
        for precision in ["float64", "float32", "float16"]:
            r = runs[precision]
            rows.append({
                "equation": r.equation,
                "Re": re_value,
                "precision": r.precision,
                "reference_type": r.reference_type,
                "dt_min": r.dt_min,
                "dt_max": r.dt_max,
                "nsteps": r.nsteps,
                "L2_error_vs_reference": r.l2_error,
                "Linf_error_vs_reference": r.linf_error,
                "conservation_error_sum": r.conservation_error_sum,
                "mass_drift": r.mass_drift,
                "TV_initial": r.tv_initial,
                "TV_final": r.tv_final,
                "runtime_sec": r.runtime_sec,
                "state_bytes": r.state_bytes,
                "memory_MB": r.state_bytes / (1024.0 ** 2),
                "est_flops_total": r.est_flops_total,
                "est_bytes_total": r.est_bytes_total,
                "stable": r.stable,
                "notes": r.notes,
            })
    return pd.DataFrame(rows)


## 11. Plotting Utilities
Generate profile plots, L2 error bars, accuracy metrics, Burgers Re sweeps, conservation/TV,
runtime/memory, computational cost, stability CFL, and table PNGs for each experiment.


In [ ]:
def dataframe_to_table_png(df: pd.DataFrame, title: str, out_path: Path) -> None:
    if df.empty:
        return
    show_df = df.copy()
    for col in show_df.columns:
        if np.issubdtype(show_df[col].dtype, np.number):
            show_df[col] = show_df[col].map(
                lambda v: f"{v:.3e}" if abs(v) < 1e4 and abs(v) > 0 else (f"{v:.2f}" if abs(v) >= 1e4 else "0")
            )
    fig, ax = plt.subplots(figsize=(max(8, 1.2 * len(show_df.columns)), max(2.8, 0.6 * (len(show_df) + 2))))
    ax.axis("off")
    ax.set_title(title)
    tbl = ax.table(cellText=show_df.values, colLabels=show_df.columns, loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1.0, 1.2)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def plot_linear_profiles(exp_dir: Path, linear_results: Dict[str, RunResult]) -> None:
    fig, ax = plt.subplots()
    x = linear_results["float64"].x
    ax.plot(x, linear_results["float64"].u_ref, linewidth=2.4, label="Exact")
    ax.plot(x, linear_results["float64"].u_final, linewidth=1.8, label="FP64")
    ax.plot(x, linear_results["float32"].u_final, linewidth=1.8, label="FP32")
    ax.plot(x, linear_results["float16"].u_final, linewidth=1.8, label="FP16")
    ax.set_title("Linear advection-diffusion: final profiles")
    ax.set_xlabel("x")
    ax.set_ylabel("u(x,t)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(exp_dir / "linear_profiles.png", dpi=220)
    plt.close(fig)


def plot_linear_error_bar(exp_dir: Path, linear_results: Dict[str, RunResult]) -> None:
    fig, ax = plt.subplots()
    labels = ["FP64", "FP32", "FP16"]
    values = [
        linear_results["float64"].l2_error,
        linear_results["float32"].l2_error,
        linear_results["float16"].l2_error,
    ]
    ax.bar(labels, values)
    ax.set_title("Linear advection-diffusion: L2 error")
    ax.set_ylabel("L2 error vs exact solution")
    fig.tight_layout()
    fig.savefig(exp_dir / "linear_error_bar.png", dpi=220)
    plt.close(fig)


def plot_linear_accuracy_metrics(exp_dir: Path, linear_results: Dict[str, RunResult]) -> None:
    labels = ["FP64", "FP32", "FP16"]
    amp = [linear_results[k].amplitude_error for k in ["float64", "float32", "float16"]]
    phase = [linear_results[k].phase_error for k in ["float64", "float32", "float16"]]

    x = np.arange(len(labels))
    w = 0.35

    fig, ax = plt.subplots()
    ax.bar(x - w / 2, amp, width=w, label="Amplitude error")
    ax.bar(x + w / 2, phase, width=w, label="Phase error")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title("Linear accuracy metrics")
    ax.legend()
    fig.tight_layout()
    fig.savefig(exp_dir / "linear_accuracy_metrics.png", dpi=220)
    plt.close(fig)


def plot_burgers_profiles_re_sweep(exp_dir: Path, burgers_results: Dict[float, Dict[str, RunResult]]) -> None:
    re_list = sorted(burgers_results.keys())
    fig, axes = plt.subplots(len(re_list), 1, figsize=(8, 3.2 * len(re_list)), sharex=True)

    if len(re_list) == 1:
        axes = [axes]

    for ax, re_value in zip(axes, re_list):
        runs = burgers_results[re_value]
        x = runs["float64"].x
        ax.plot(x, runs["float64"].u_ref, linewidth=2.2, label="Reference")
        ax.plot(x, runs["float64"].u_final, linewidth=1.8, label="FP64")
        ax.plot(x, runs["float32"].u_final, linewidth=1.8, label="FP32")
        ax.plot(x, runs["float16"].u_final, linewidth=1.8, label="FP16")
        ax.set_title(f"Viscous Burgers final profile, Re = {int(re_value)}")
        ax.set_ylabel("u")
        ax.legend(loc="best", fontsize=8)

    axes[-1].set_xlabel("x")
    fig.tight_layout()
    fig.savefig(exp_dir / "burgers_profiles_re_sweep.png", dpi=220)
    plt.close(fig)


def plot_burgers_error_vs_re(exp_dir: Path, burgers_results: Dict[float, Dict[str, RunResult]]) -> None:
    fig, ax = plt.subplots()

    re_values = sorted(burgers_results.keys())
    fp32_errors = [burgers_results[re]["float32"].l2_error for re in re_values]
    fp16_errors = [burgers_results[re]["float16"].l2_error for re in re_values]

    ax.plot(re_values, fp32_errors, marker="o", linewidth=2.0, label="FP32 vs reference")
    ax.plot(re_values, fp16_errors, marker="s", linewidth=2.0, label="FP16 vs reference")
    ax.set_xscale("log")
    positive_errors = [v for v in fp32_errors + fp16_errors if np.isfinite(v) and v > 0.0]
    ax.set_yscale("log" if positive_errors else "linear")
    ax.set_title("Burgers: L2 error vs Reynolds number")
    ax.set_xlabel("Re")
    ax.set_ylabel("L2 error")
    ax.legend()
    fig.tight_layout()
    fig.savefig(exp_dir / "burgers_error_vs_re.png", dpi=220)
    plt.close(fig)


def plot_ultra_high_re_front_zoom(exp_dir: Path, burgers_results: Dict[float, Dict[str, RunResult]]) -> None:
    if not burgers_results:
        return
    re_max = max(burgers_results.keys())
    if re_max < 100000:
        return

    runs = burgers_results[re_max]
    x = runs["float64"].x
    u_ref = runs["float64"].u_ref

    grad = np.abs(np.gradient(u_ref, x))
    idx = int(np.argmax(grad))
    x0 = x[idx]
    half_window = 0.08
    dist = np.abs(((x - x0 + 0.5) % 1.0) - 0.5)
    mask = dist <= half_window

    fig, ax = plt.subplots()
    ax.plot(x[mask], u_ref[mask], linewidth=2.4, label="Reference")
    ax.plot(x[mask], runs["float64"].u_final[mask], linewidth=1.8, label="FP64")
    ax.plot(x[mask], runs["float32"].u_final[mask], linewidth=1.8, label="FP32")
    ax.plot(x[mask], runs["float16"].u_final[mask], linewidth=1.8, label="FP16")
    ax.set_title(f"Ultra-high-Re front zoom (Re={int(re_max)})")
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.legend()
    fig.tight_layout()
    fig.savefig(exp_dir / "ultra_high_re_front_zoom.png", dpi=220)
    plt.close(fig)


def plot_conservation_error(
    exp_dir: Path,
    linear_results: Optional[Dict[str, RunResult]],
    burgers_results: Optional[Dict[float, Dict[str, RunResult]]],
) -> None:
    panels = (1 if linear_results is not None else 0) + (1 if burgers_results is not None else 0)
    fig, axes = plt.subplots(1, panels, figsize=(5.5 * panels, 4.6))
    if panels == 1:
        axes = np.array([axes])

    idx = 0
    if linear_results is not None:
        labels = ["FP64", "FP32", "FP16"]
        vals = [linear_results[k].conservation_error_sum for k in ["float64", "float32", "float16"]]
        axes[idx].bar(labels, vals)
        axes[idx].set_yscale("log")
        axes[idx].set_title("Linear conservation error")
        idx += 1

    if burgers_results is not None:
        re_max = max(burgers_results.keys())
        labels = ["FP64", "FP32", "FP16"]
        vals = [burgers_results[re_max][k].conservation_error_sum for k in ["float64", "float32", "float16"]]
        axes[idx].bar(labels, vals)
        axes[idx].set_yscale("log")
        axes[idx].set_title(f"Burgers conservation error (Re={int(re_max)})")

    fig.tight_layout()
    fig.savefig(exp_dir / "conservation_error.png", dpi=220)
    plt.close(fig)


def plot_total_variation(
    exp_dir: Path,
    linear_results: Optional[Dict[str, RunResult]],
    burgers_results: Optional[Dict[float, Dict[str, RunResult]]],
) -> None:
    panels = (1 if linear_results is not None else 0) + (1 if burgers_results is not None else 0)
    fig, axes = plt.subplots(1, panels, figsize=(5.5 * panels, 4.6))
    if panels == 1:
        axes = np.array([axes])

    idx = 0
    if linear_results is not None:
        labels = ["FP64", "FP32", "FP16"]
        vals = [linear_results[k].tv_final for k in ["float64", "float32", "float16"]]
        axes[idx].bar(labels, vals)
        axes[idx].set_title("Linear total variation")
        idx += 1

    if burgers_results is not None:
        re_max = max(burgers_results.keys())
        labels = ["FP64", "FP32", "FP16"]
        vals = [burgers_results[re_max][k].tv_final for k in ["float64", "float32", "float16"]]
        axes[idx].bar(labels, vals)
        axes[idx].set_title(f"Burgers total variation (Re={int(re_max)})")

    fig.tight_layout()
    fig.savefig(exp_dir / "total_variation.png", dpi=220)
    plt.close(fig)


def plot_runtime_comparison(
    exp_dir: Path,
    linear_results: Optional[Dict[str, RunResult]],
    burgers_results: Optional[Dict[float, Dict[str, RunResult]]],
) -> None:
    panels = (1 if linear_results is not None else 0) + (1 if burgers_results is not None else 0)
    fig, axes = plt.subplots(1, panels, figsize=(5.5 * panels, 4.6))
    if panels == 1:
        axes = np.array([axes])

    idx = 0
    labels = ["FP64", "FP32", "FP16"]

    if linear_results is not None:
        vals = [linear_results[k].runtime_sec for k in ["float64", "float32", "float16"]]
        axes[idx].bar(labels, vals)
        axes[idx].set_title("Linear case runtime")
        axes[idx].set_ylabel("seconds")
        idx += 1

    if burgers_results is not None:
        re_max = max(burgers_results.keys())
        vals = [burgers_results[re_max][k].runtime_sec for k in ["float64", "float32", "float16"]]
        axes[idx].bar(labels, vals)
        axes[idx].set_title(f"Burgers runtime at Re={int(re_max)}")
        axes[idx].set_ylabel("seconds")

    fig.tight_layout()
    fig.savefig(exp_dir / "runtime_comparison.png", dpi=220)
    plt.close(fig)


def plot_computational_cost(
    exp_dir: Path,
    linear_results: Optional[Dict[str, RunResult]],
    burgers_results: Optional[Dict[float, Dict[str, RunResult]]],
) -> None:
    panels = (1 if linear_results is not None else 0) + (1 if burgers_results is not None else 0)
    fig, axes = plt.subplots(1, panels, figsize=(5.5 * panels, 4.6))
    if panels == 1:
        axes = np.array([axes])

    idx = 0
    labels = ["FP64", "FP32", "FP16"]

    if linear_results is not None:
        vals = [linear_results[k].state_bytes / (1024.0 ** 2) for k in ["float64", "float32", "float16"]]
        axes[idx].bar(labels, vals)
        axes[idx].set_title("Linear memory usage")
        axes[idx].set_ylabel("MB")
        idx += 1

    if burgers_results is not None:
        re_max = max(burgers_results.keys())
        vals = [burgers_results[re_max][k].state_bytes / (1024.0 ** 2) for k in ["float64", "float32", "float16"]]
        axes[idx].bar(labels, vals)
        axes[idx].set_title(f"Burgers memory usage (Re={int(re_max)})")
        axes[idx].set_ylabel("MB")

    fig.tight_layout()
    fig.savefig(exp_dir / "computational_cost.png", dpi=220)
    plt.close(fig)


def plot_stability_cfl(exp_dir: Path, stability_df: pd.DataFrame) -> None:
    if stability_df.empty:
        return

    fig, ax = plt.subplots()
    eqs = stability_df["equation"].unique().tolist()
    labels = ["FP64", "FP32", "FP16"]
    width = 0.35
    x = np.arange(len(labels))

    for j, eq in enumerate(eqs):
        df = stability_df[stability_df["equation"] == eq]
        vals = [
            float(df[df["precision"] == p]["largest_stable_cfl"].iloc[0])
            for p in ["float64", "float32", "float16"]
        ]
        ax.bar(x + (j - (len(eqs) - 1) / 2) * width, vals, width=width, label=eq)

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title("Largest stable CFL by precision")
    ax.set_ylabel("largest stable CFL")
    ax.legend()

    fig.tight_layout()
    fig.savefig(exp_dir / "stability_cfl.png", dpi=220)
    plt.close(fig)


## 12. Experiment Catalogue
Define all experiments via `build_experiments()`. Each `ExperimentConfig` specifies
the grid, final times, Re values, CFL, and which solver phases are active.


In [ ]:
def build_experiments() -> List[ExperimentConfig]:
    return [
        ExperimentConfig(
            name="exp01_baseline_regime_sweep",
            description=(
                "Experiment 01 - Baseline regime sweep / most accurate regime. "
                "Reproduces the original Scheme 3 plots under an experiment head. "
                "Linear smooth test plus Burgers at Re=10,100,1000."
            ),
            nx=1024,
            tfinal_linear=0.20,
            nu_linear=1.0e-3,
            linear_amplitude=1.0,
            tfinal_burgers=0.12,
            re_values=[10.0, 100.0, 1000.0],
            cfl_total=0.80,
            run_linear=True,
            run_burgers=True,
            run_stability_sweep=True,
            stability_re_for_burgers=1000.0,
        ),
        ExperimentConfig(
            name="exp02_ultra_high_re_shock_stress",
            description=(
                "Experiment 02 - Ultra-high-Re shock stress. "
                "Adds Re=100000 to test convection-dominated sharpening and precision sensitivity. "
                "Includes an ultra-high-Re front-zoom plot for intuitive comparison."
            ),
            nx=1024,
            tfinal_linear=0.20,
            nu_linear=1.0e-3,
            linear_amplitude=1.0,
            tfinal_burgers=0.12,
            re_values=[10.0, 100.0, 1000.0, 100000.0],
            cfl_total=0.80,
            run_linear=True,
            run_burgers=True,
            run_stability_sweep=True,
            stability_re_for_burgers=100000.0,
        ),
        ExperimentConfig(
            name="exp03_underresolved_high_re_breakdown_nx128",
            description=(
                "Experiment 03 - Edge case: under-resolved shock breakdown. "
                "Re=100000 on a coarse grid nx=128. "
                "Designed to expose a regime where FP32 and FP16 can both degrade strongly or fail."
            ),
            nx=128,
            tfinal_linear=0.20,
            nu_linear=1.0e-3,
            linear_amplitude=1.0,
            tfinal_burgers=0.12,
            re_values=[100000.0],
            cfl_total=0.80,
            run_linear=False,
            run_burgers=True,
            run_stability_sweep=True,
            stability_re_for_burgers=100000.0,
        ),
        ExperimentConfig(
            name="exp04_tiny_amplitude_quantization_stress",
            description=(
                "Experiment 04 - Edge case: tiny-amplitude quantization stress. "
                "Very small-amplitude linear wave to expose precision floor and loss of small updates."
            ),
            nx=1024,
            tfinal_linear=0.20,
            nu_linear=1.0e-3,
            linear_amplitude=1.0e-6,
            tfinal_burgers=0.12,
            re_values=[],
            cfl_total=0.80,
            run_linear=True,
            run_burgers=False,
            run_stability_sweep=True,
        ),
        ExperimentConfig(
            name="exp05_long_horizon_drift_accumulation",
            description=(
                "Experiment 05 - Edge case: long-horizon drift accumulation. "
                "Extended final time to accumulate conservation and TV drift due to precision."
            ),
            nx=1024,
            tfinal_linear=1.00,
            nu_linear=1.0e-3,
            linear_amplitude=1.0,
            tfinal_burgers=0.30,
            re_values=[10.0, 100.0, 1000.0],
            cfl_total=0.50,
            run_linear=True,
            run_burgers=True,
            run_stability_sweep=False,
        ),
        ExperimentConfig(
            name="exp06_cfl_overdrive_method_failure",
            description=(
                "Experiment 06 - Edge case: method-level failure / overdriven CFL. "
                "Very high Re with intentionally aggressive CFL to test when the method itself breaks, "
                "independent of only precision."
            ),
            nx=256,
            tfinal_linear=0.20,
            nu_linear=1.0e-3,
            linear_amplitude=1.0,
            tfinal_burgers=0.12,
            re_values=[100000.0],
            cfl_total=1.20,
            run_linear=False,
            run_burgers=True,
            run_stability_sweep=False,
        ),
    ]


## 13. Run All Experiments and Save Summaries
`run_experiment` orchestrates the full pipeline per experiment: baselines → linear solver →
Burgers solver → stability sweep → plots → summary tables. The batch loop catches failures
per-experiment, logs full tracebacks to disk, and saves a consolidated summary CSV.


In [ ]:
def run_experiment(cfg: ExperimentConfig) -> pd.DataFrame:
    device = pick_device(cfg)
    exp_dir = OUTDIR / cfg.name
    exp_dir.mkdir(parents=True, exist_ok=True)

    _, linear_exact, burgers_refs = build_baselines(cfg, exp_dir / "baselines")

    linear_results = None
    burgers_results = None

    if cfg.run_linear:
        linear_results = {
            p: run_linear_advection_diffusion(p, cfg, device, linear_exact)
            for p in ["float64", "float32", "float16"]
        }
        linear_df = linear_summary_table(linear_results)
        linear_df.to_csv(exp_dir / "linear_summary.csv", index=False)

        plot_linear_profiles(exp_dir, linear_results)
        plot_linear_error_bar(exp_dir, linear_results)
        plot_linear_accuracy_metrics(exp_dir, linear_results)
        dataframe_to_table_png(
            linear_df[["precision", "L2_error", "Linf_error", "amplitude_error", "phase_error"]],
            "Table: Linear accuracy",
            exp_dir / "table_linear_accuracy.png",
        )
    else:
        linear_df = pd.DataFrame()

    if cfg.run_burgers:
        burgers_results = {}
        for re_value in cfg.re_values:
            dt_schedule, pilot_stable = build_burgers_dt_schedule_fp64(
                re_value, cfg, device
            )
            if not pilot_stable:
                raise RuntimeError(
                    f"Pilot schedule unstable for Re={re_value} in experiment {cfg.name}"
                )

            burgers_results[re_value] = {
                p: run_burgers_with_schedule(
                    p, re_value, dt_schedule, cfg, device, burgers_refs[re_value]
                )
                for p in ["float64", "float32", "float16"]
            }

        burgers_df = burgers_summary_table(burgers_results)
        burgers_df.to_csv(exp_dir / "burgers_summary.csv", index=False)

        plot_burgers_profiles_re_sweep(exp_dir, burgers_results)
        plot_burgers_error_vs_re(exp_dir, burgers_results)
        plot_ultra_high_re_front_zoom(exp_dir, burgers_results)
        dataframe_to_table_png(
            burgers_df[["Re", "precision", "L2_error_vs_reference", "Linf_error_vs_reference"]],
            "Table: Burgers accuracy",
            exp_dir / "table_burgers_accuracy.png",
        )
    else:
        burgers_df = pd.DataFrame()

    plot_conservation_error(exp_dir, linear_results, burgers_results)
    plot_total_variation(exp_dir, linear_results, burgers_results)
    plot_runtime_comparison(exp_dir, linear_results, burgers_results)
    plot_computational_cost(exp_dir, linear_results, burgers_results)

    cost_frames = []
    if not linear_df.empty:
        tmp = linear_df[["precision", "runtime_sec", "memory_MB", "est_flops_total", "est_bytes_total"]].copy()
        tmp.insert(0, "equation", "linear")
        cost_frames.append(tmp)
    if not burgers_df.empty:
        re_max = max(cfg.re_values) if cfg.re_values else None
        tmp = burgers_df[burgers_df["Re"] == re_max][["precision", "runtime_sec", "memory_MB", "est_flops_total", "est_bytes_total"]].copy() if re_max is not None else pd.DataFrame()
        if not tmp.empty:
            tmp.insert(0, "equation", f"burgers_Re_{int(re_max)}")
            cost_frames.append(tmp)
    if cost_frames:
        cost_df = pd.concat(cost_frames, ignore_index=True)
        dataframe_to_table_png(cost_df, "Table: Computational cost", exp_dir / "table_computational_cost.png")

    if cfg.run_stability_sweep:
        u_ref_burg = None
        if cfg.run_burgers and cfg.stability_re_for_burgers in burgers_refs:
            u_ref_burg = burgers_refs[cfg.stability_re_for_burgers]

        stability_df = save_stability_table(
            cfg, device, exp_dir, linear_exact, u_ref_burg
        )
        plot_stability_cfl(exp_dir, stability_df)
        dataframe_to_table_png(stability_df, "Table: Stability", exp_dir / "table_stability.png")
    else:
        stability_df = pd.DataFrame()

    parts = []
    if not linear_df.empty:
        tmp = linear_df.copy()
        tmp.insert(0, "experiment", cfg.name)
        parts.append(tmp)

    if not burgers_df.empty:
        tmp = burgers_df.copy()
        tmp.insert(0, "experiment", cfg.name)
        parts.append(tmp)

    if not stability_df.empty:
        tmp = stability_df.copy()
        tmp.insert(0, "experiment", cfg.name)
        parts.append(tmp)

    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


all_frames = []
failed_runs = []

print("=" * 90)
print("Scheme 3 experiment suite")
print("=" * 90)

for cfg in build_experiments():
    print(f"Running {cfg.name}")
    try:
        df = run_experiment(cfg)
        all_frames.append(df)
        print(f"Completed {cfg.name}")
    except Exception as exc:
        failed_runs.append({
            "experiment": cfg.name,
            "error": str(exc),
            "traceback": traceback.format_exc(),
        })
        print(f"FAILED {cfg.name}: {exc}")

all_df = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
all_df.to_csv(OUTDIR / "all_experiments_summary.csv", index=False)

if failed_runs:
    failed_df = pd.DataFrame([{k: v for k, v in item.items() if k != "traceback"} for item in failed_runs])
    failed_df.to_csv(OUTDIR / "failed_experiments.csv", index=False)
    with open(OUTDIR / "failed_experiments.txt", "w", encoding="utf-8") as f:
        for item in failed_runs:
            f.write(f"Experiment: {item['experiment']}\n")
            f.write(f"Error: {item['error']}\n")
            f.write(item["traceback"])
            f.write("\n" + ("-" * 90) + "\n")

print("\nSaved:")
print(f"  {OUTDIR / 'all_experiments_summary.csv'}")
if failed_runs:
    print(f"  {OUTDIR / 'failed_experiments.csv'}")
print("\nDone.")

all_df


Scheme 3 experiment suite
Running exp01_baseline_regime_sweep


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/_core/_methods.py:134: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


Completed exp01_baseline_regime_sweep
Running exp02_ultra_high_re_shock_stress


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/numpy/_core/_methods.py:134: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


Completed exp02_ultra_high_re_shock_stress
Running exp03_underresolved_high_re_breakdown_nx128
Completed exp03_underresolved_high_re_breakdown_nx128
Running exp04_tiny_amplitude_quantization_stress
Completed exp04_tiny_amplitude_quantization_stress
Running exp05_long_horizon_drift_accumulation
Completed exp05_long_horizon_drift_accumulation
Running exp06_cfl_overdrive_method_failure
Completed exp06_cfl_overdrive_method_failure

Saved:
  /Users/rajneesh/Downloads/DS_289_NSDE_Project_Report_Template/Schene_3_Final/all_experiments_summary.csv

Done.


,experiment,equation,precision,reference_type,dt_last,nsteps,L2_error,Linf_error,amplitude_error,phase_error,...,est_flops_total,est_bytes_total,stable,notes,Re,dt_min,dt_max,L2_error_vs_reference,Linf_error_vs_reference,largest_stable_cfl
0,exp01_baseline_regime_sweep,linear_advection_diffusion,float64,exact_solution,0.000256,781.0,0.000004,0.000005,7.158311e-09,0.000000,...,1.919386e+07,5.758157e+07,True,OK,NaN,NaN,NaN,NaN,NaN,NaN
1,exp01_baseline_regime_sweep,linear_advection_diffusion,float32,exact_solution,0.000256,781.0,0.000004,0.000005,4.151925e-07,0.000000,...,1.919386e+07,2.879078e+07,True,OK,NaN,NaN,NaN,NaN,NaN,NaN
2,exp01_baseline_regime_sweep,linear_advection_diffusion,float16,exact_solution,0.000256,781.0,0.093236,0.174065,9.613856e-02,0.006836,...,1.919386e+07,1.439539e+07,True,OK,NaN,NaN,NaN,NaN,NaN,NaN
3,exp01_baseline_regime_sweep,viscous_burgers,float64,cole_hopf_reference,NaN,31580.0,NaN,NaN,NaN,NaN,...,1.552220e+09,3.880550e+09,True,OK,10.0,0.000001,0.000004,0.000002,0.000002,NaN
4,exp01_baseline_regime_sweep,viscous_burgers,float32,cole_hopf_reference,NaN,31580.0,NaN,NaN,NaN,NaN,...,1.552220e+09,1.940275e+09,True,OK,10.0,0.000001,0.000004,0.000013,0.000024,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,exp05_long_horizon_drift_accumulation,viscous_burgers,float32,cole_hopf_reference,NaN,1868.0,NaN,NaN,NaN,NaN,...,9.181594e+07,1.147699e+08,True,OK,1000.0,0.000097,0.000163,0.539124,0.963935,NaN
62,exp05_long_horizon_drift_accumulation,viscous_burgers,float16,cole_hopf_reference,NaN,1868.0,NaN,NaN,NaN,NaN,...,9.181594e+07,5.738496e+07,True,OK,1000.0,0.000097,0.000163,0.525638,0.871668,NaN
63,exp06_cfl_overdrive_method_failure,viscous_burgers,float64,cole_hopf_reference,NaN,26.0,NaN,NaN,NaN,NaN,...,3.194880e+05,7.987200e+05,True,OK,100000.0,0.003403,0.004664,0.707018,1.000552,NaN
64,exp06_cfl_overdrive_method_failure,viscous_burgers,float32,cole_hopf_reference,NaN,26.0,NaN,NaN,NaN,NaN,...,3.194880e+05,3.993600e+05,True,OK,100000.0,0.003403,0.004664,0.707185,1.069387,NaN
